In [2]:
!pip install -q transformers datasets evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [3]:
!pip install -q pandas numpy matplotlib

In [4]:
!pip install -q wandb


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.json to test.json
Saving train.json to train.json
Saving valid.json to valid.json


In [6]:
import pandas as pd
import numpy as np
import evaluate

In [7]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: maabdullatif1 (maabdullatif1-king-faisal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
data = pd.read_json('train.json')

In [9]:
data['comment_type'].value_counts()

,count
comment_type,
Summary,8398


In [10]:
data['label'].value_counts()

,count
label,
1,4199
0,4199


In [11]:
val = pd.read_json('valid.json')[['new_comment_raw', 'new_code_raw', 'label']]
test = pd.read_json('test.json')[['new_comment_raw', 'new_code_raw', 'label']]
val.head()

,new_comment_raw,new_code_raw,label
0,Return SQL selector query for getting tasks wi...,public static QueryTemplate queryTempl...,1
1,Return period of ghost connections cleanup tas...,public int getGhostConnsCleanupPeriod() {\...,0
2,Allocates an initialized and initially unlocke...,\tpublic static SpinLock allocateSpinLock() {\...,1
3,Rebalance a frame for load balancing,private static Frame reBalance(final Frame f...,0
4,Sets the host of the proxy.,public Proxy setHost( String host )\n {...,1


In [12]:
train = data[['new_comment_raw', 'new_code_raw', 'label']]

Why should we combine them together (code and comment) ??

* CodeBERT isn't designed to process two independent inputs in parallel.

* CodeBERT was trained with both code and natural language (NL) jointly, using the format:

    `[CLS] code tokens [SEP] comment tokens [SEP]`

* When we concatenate them, CodeBERT can learn relationships between the code and comment tokens.



In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [14]:
from datasets import Dataset
from datasets import DatasetDict

train = Dataset.from_pandas(train)
test = Dataset.from_pandas(test)
val = Dataset.from_pandas(val)

dataset = DatasetDict({
    "train": train,
    "validation": val,
    "test": test
})

In [15]:
def preprocess_function(examples):
    return tokenizer(
        examples["new_code_raw"],
        examples["new_comment_raw"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [16]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/8398 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8398
    })
    validation: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1034
    })
    test: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [18]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [19]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall": recall.compute(predictions=preds, references=labels)["recall"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"]
    }


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-712792143.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.484000,0.306786,0.860735,0.997333,0.723404,0.838565
2,0.385900,0.306103,0.856867,0.901961,0.800774,0.848361
3,0.272100,0.351762,0.844294,0.850394,0.835590,0.842927
4,0.176800,0.524489,0.855899,0.878601,0.825919,0.851446
5,0.099800,0.756596,0.850097,0.863454,0.831721,0.847291


TrainOutput(global_step=2625, training_loss=0.2737720235188802, metrics={'train_runtime': 4184.7584, 'train_samples_per_second': 10.034, 'train_steps_per_second': 0.627, 'total_flos': 1.10480332145664e+16, 'train_loss': 0.2737720235188802, 'epoch': 5.0})

In [ ]:
import wandb
test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
print(test_results)

wandb.log(test_results)

print("Metrics logged to run:", wandb.run.name, "(id:", wandb.run.id, ")")
wandb.finish()

{'eval_loss': 0.32664957642555237, 'eval_accuracy': 0.850844277673546, 'eval_precision': 0.9973404255319149, 'eval_recall': 0.7035647279549718, 'eval_f1': 0.8250825082508251, 'eval_runtime': 30.4958, 'eval_samples_per_second': 34.956, 'eval_steps_per_second': 2.197, 'epoch': 5.0}
Metrics logged to run: pious-music-9 (id: yhydgqtp )


epoch,▁
eval/accuracy,█▆▁▆▃▄
eval/f1,▅▇▆█▇▁
eval/loss,▁▁▂▄█▁
eval/precision,█▃▁▂▂█
eval/recall,▂▆█▇█▁
eval/runtime,▁▁▁▁▁█
eval/samples_per_second,█▇███▁
eval/steps_per_second,█▇███▁
eval_accuracy,▁
+12,...
